# Save the Model

To save this model so that you can use it from various locations, including other notebooks or the model server, upload it to s3-compatible storage.

## Install the required packages and define a function for the upload

In [1]:
!pip install boto3==1.35.55 botocore==1.35.55

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cuda12.9-ubi9/simple/
ERROR: Could not find a version that satisfies the requirement boto3==1.35.55 (from versions: 1.35.42, 1.40.61, 1.42.19, 1.42.54, 1.42.55, 1.42.56, 1.42.57, 1.42.58, 1.42.59, 1.42.60, 1.42.61, 1.42.62, 1.42.63, 1.42.64, 1.42.65, 1.42.66, 1.42.67, 1.42.68, 1.42.69, 1.42.70, 1.42.71, 1.42.72, 1.42.73, 1.42.75, 1.42.76, 1.42.77, 1.42.78, 1.42.79, 1.42.80, 1.42.81, 1.42.84, 1.42.85, 1.42.86, 1.42.87, 1.42.88, 1.42.89)
ERROR: No matching distribution found for boto3==1.35.55


In [2]:
%pip install boto3 botocore --extra-index-url https://pypi.org/simple/

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cuda12.9-ubi9/simple/, https://pypi.org/simple/
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import boto3
import botocore

aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError("One or more connection variables are empty.  "
                     "Please check your connection to an S3 bucket.")

session = boto3.session.Session(aws_access_key_id=aws_access_key_id,
                                aws_secret_access_key=aws_secret_access_key)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name)

bucket = s3_resource.Bucket(bucket_name)


def upload_directory_to_s3(local_directory, s3_prefix):
    num_files = 0
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"{file_path} -> {s3_key}")
            bucket.upload_file(file_path, s3_key)
            num_files += 1
    return num_files


def list_objects(prefix):
    filter = bucket.objects.filter(Prefix=prefix)
    for obj in filter.all():
        print(obj.key)

## Verify the upload

In your S3 bucket, under the `models` upload prefix, run the `list_object` command. As best practice, to avoid mixing up model files, keep only one model and its required files in a given prefix or directory. This practice allows you to download and serve a directory with all the files that a model requires. 

If this is the first time running the code, this cell will have no output.

If you've already uploaded your model, you should see this output: `models/fraud/1/model.onnx`


In [4]:
list_objects("models")

## Upload and check again

Use the function to upload the `models` folder in a recursive fashion:

In [5]:
local_models_directory = "models"

if not os.path.isdir(local_models_directory):
    raise ValueError(f"The directory '{local_models_directory}' does not exist.  "
                     "Did you finish training the model in the previous notebook?")

num_files = upload_directory_to_s3("models", "models")

if num_files == 0:
    raise ValueError("No files uploaded.  Did you finish training and "
                     "saving the model to the \"models\" directory?  "
                     "Check for \"models/fraud/1/model.onnx\"")


models/fraud/1/model.onnx -> models/fraud/1/model.onnx


To confirm this worked, run the `list_objects` function again:

In [7]:
list_objects("models")

models/fraud/1/model.onnx


### Next Step

Now that you've saved the model to s3 storage, you can refer to the model by using the same data connection to serve the model as an API.
